<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Many metrics at once

**[Many metrics at once](https://learning.nextia-ai.com/courses/anomaly-detection/m05/many-metrics-at-once/)** · Anomaly Detection: Find What Is Unusual · Module 5, lesson 5 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** compare one detector per metric with one detector for all metrics on the ticket API, and group alerts so that one incident gives one alert, not forty.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (about 3 MB). No account. |
| **You should know** | Seasonal baselines, from [Normal changes with the hour and the season](https://learning.nextia-ai.com/courses/anomaly-detection/m05/normal-changes-with-the-hour-and-the-season/); Isolation Forest, from [Isolation Forest](https://learning.nextia-ai.com/courses/anomaly-detection/m04/isolation-forest/). |
| **You do not need** | A local project. The setup below downloads everything. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m05/many-metrics-at-once/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M05-L05-many-metrics-at-once/many-metrics-at-once-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`ticket_api_metrics.csv.gz`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py`, the course's shared code. It needs an internet connection and no account. It takes a few seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "ticket_api_metrics.csv.gz": "73b420194586f72b760fd672f89ee2526035385965668a3098cf1e34dfa123c4",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. One incident, many alerts

A smoke alarm, a heat alarm and a gas alarm can all ring for one kitchen fire. The fire brigade needs **one** call, not three calls every minute.

Run the next cell. Three detectors watch 12 minutes. Each 1 is an alert. Then we join the minutes with an alert that are less than 5 minutes apart into **events**.

> **Check.** You should see 10 raw alerts, 6 minutes with an alert, and 2 events.

In [ ]:
# Twelve minutes, three detectors. 1 = the detector alerts in that minute.
minutes = pd.date_range("2025-10-08 14:00", periods=12, freq="1min")
flags = pd.DataFrame({"latency": [0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1],
                      "errors":  [0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
                      "requests": [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]}, index=minutes)
raw_alerts = int(flags.to_numpy().sum())                      # one alert per detector and minute
any_alert = flags.any(axis=1)
def group(times, gap):                                         # join alerts less than `gap` apart
    events, start, last = [], None, None
    for t in times:
        if start is None or t - last > gap:
            if start is not None:
                events.append((start, last))
            start = t
        last = t
    if start is not None:
        events.append((start, last))
    return events
events = group(minutes[any_alert], pd.Timedelta("5min"))
print(flags.set_axis(minutes.strftime("%H:%M")).T.to_string())
print("raw alerts:", raw_alerts, "| minutes with an alert:", int(any_alert.sum()), "| events (5-minute gap):", len(events))

## 2. The ticket API's three metrics

The next cell loads 60 days of the ticket API, one row per minute: the requests, the share of requests that failed (`error_rate`) and the p95 latency (95% of the requests were faster than this).

In [ ]:
import compare

metrics = pd.read_csv("data/ticket_api_metrics.csv.gz", parse_dates=["ts"]).set_index("ts")
ts = metrics.index
print(metrics.describe().loc[["mean", "50%", "max"]].round(3))

## 3. One detector per metric

Run the next cell. Four detectors, each on one metric: a latency threshold, an error-rate threshold, the seasonal baseline of latency (the same minute 1 and 2 weeks before), and an error count test (more errors than 0.4% of the requests can explain). Each one is scored on the planted incidents of its own metric.

> **Check.** The error-rate threshold has 526 alert events in 60 days, and only 1 of them is on a planted incident.

In [ ]:
def robust_z(x, center, spread):
    return (x - center) / (1.4826 * np.maximum(spread, 1e-9))

lat = metrics.latency_p95_ms
alarms = {"latency threshold": lat > 350, "error-rate threshold": metrics.error_rate > 0.03}
week_ago = pd.concat([lat.shift(7 * 1440 * k) for k in (1, 2)], axis=1).mean(axis=1)   # same minute, 1 and 2 weeks before
dev = (lat - week_ago).rolling(15).median()
normal_week = dev.loc["2025-09-15":"2025-09-21"]
spread = (normal_week - normal_week.median()).abs().median()
alarms["seasonal baseline, latency"] = (robust_z(dev, 0, spread) > 4).fillna(False)
expected = metrics.requests * 0.004                                   # 0.4% of requests fail on a normal day
err_z = (metrics.error_rate * metrics.requests - expected) / np.sqrt(np.maximum(expected, 1e-9))
alarms["error count test"] = (err_z > 4) & (metrics.requests >= 20)
signals = {"latency threshold": "latency", "error-rate threshold": "errors",
           "seasonal baseline, latency": "latency", "error count test": "errors"}
table = compare.Table()
for name, f in alarms.items():
    table.add(compare.score_windows(ts, f, "ticket_api", name, explain_minutes=2, signal=signals[name]))
print(pd.DataFrame(table.rows)[["detector", "alert events", "false alert events", "recall", "minutes to detection"]].to_string(index=False))

The error-rate threshold finds the release spike at once, but it also alarms 525 other times. In 2,130 of its 2,736 alert minutes, only one request failed: one error in 30 requests is already 3.3%. The error count test asks a better question: are there more errors than the number of requests can explain? It has fewer false events (327), but still many.

## 4. One detector for all three metrics

> **Predict.** An Isolation Forest sees the three metrics together. Will it have more or fewer alert events than the two per-metric detectors together?

Run the next cell. It scales each metric against its usual value at that hour of day, learns an Isolation Forest on the first two weeks, and alerts on the top 0.1% of scores.

In [ ]:
from sklearn.ensemble import IsolationForest

features = pd.DataFrame({"latency": np.log(lat), "errors": metrics.error_rate, "requests": np.log1p(metrics.requests)})
by_hour = features.groupby(ts.hour)                                    # compare each minute with its hour of day
scaled = (features - by_hour.transform("median")) / by_hour.transform(lambda x: 1.4826 * (x - x.median()).abs().median() + 1e-3)
train = scaled[ts < "2025-09-15"]                                      # the first two weeks are "normal"
forest = IsolationForest(n_estimators=200, random_state=0, n_jobs=1).fit(train)
score = -forest.score_samples(scaled)
limit = np.quantile(-forest.score_samples(train), 0.999)               # the top 0.1% of the training minutes
iso = score > limit
both = alarms["latency threshold"] | alarms["error count test"]
for name, f in (("Isolation Forest, 3 metrics", iso), ("two per-metric detectors", both)):
    table.add(compare.score_windows(ts, f, "ticket_api", name, explain_minutes=5 if "Forest" in name else 2))
print(pd.DataFrame(table.rows[-2:])[["detector", "alert events", "false alert events", "recall", "minutes to detection"]].to_string(index=False))

The Isolation Forest has 61 alert events and finds all three incidents; the two per-metric detectors together have 337. One detector for all metrics sees a minute where everything is a little odd at once. But its alert says only "this minute is strange": a person must still find which metric moved, so it takes longer to explain.

## 5. Group the alerts

The next cell counts the raw alert minutes of the four per-metric detectors, and groups them into events with a 30-minute gap.

In [ ]:
any_alarm = alarms["latency threshold"] | alarms["error-rate threshold"] | alarms["seasonal baseline, latency"] | alarms["error count test"]
spike = (ts >= "2025-10-08 14:05") & (ts < "2025-10-08 14:45")
grouped = compare._events(ts, any_alarm, pd.Timedelta("30min"))
print("alert minutes, four detectors:", int(any_alarm.sum()))
print("alert minutes in the release spike:", int((any_alarm & spike).sum()))
print("events after grouping (30 minutes):", len(grouped))
print("events that overlap the spike:", sum(1 for s, e in grouped if s <= pd.Timestamp("2025-10-08 14:45") and e >= pd.Timestamp("2025-10-08 14:05")))

The chart shows the release spike: latency, error rate and requests from 13:30 to 15:30 on 2025-10-08. The orange band is the planted spike.

In [ ]:
part = slice("2025-10-08 13:30", "2025-10-08 15:29")
fig, axes = plt.subplots(3, 1, figsize=(10, 6), sharex=True)
for ax, col in zip(axes, ["latency_p95_ms", "error_rate", "requests"]):
    ax.plot(metrics.loc[part].index, metrics.loc[part, col], lw=0.8, color="grey")
    ax.set_ylabel(col)
    ax.axvspan(pd.Timestamp("2025-10-08 14:05"), pd.Timestamp("2025-10-08 14:45"), color="orange", alpha=0.2)
plt.tight_layout()
plt.show()

## 6. Check yourself

**Your turn 1.** Group the same alert minutes (`any_alarm`) with a gap of **2 hours** instead of 30 minutes. How many events are there? Write it as `events_2h`.

In [ ]:
events_2h = ...   # your code

In [ ]:
expect('events_2h', events_2h, 113)

**Your turn 2.** How many minutes after the start of the memory leak does the Isolation Forest's first event come? Read it from its row in `table`, and write it as `iso_leak_minutes`.

In [ ]:
iso_leak_minutes = ...   # your code

In [ ]:
expect('iso_leak_minutes', iso_leak_minutes, 2752)

## Recap

- One detector per metric is simple to explain, but every detector sends its own alerts for the same incident.
- One detector for all metrics sees minutes where everything is a little odd; its alerts take longer to explain.
- Group alerts that are close in time into one event. The gap is a choice: too short, and one incident is many events; too long, and two incidents become one.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Autoencoders and reconstruction error](https://learning.nextia-ai.com/courses/anomaly-detection/m06/autoencoders-and-reconstruction-error/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m05/many-metrics-at-once/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.